In [ ]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_log_error as RMSE
from itertools import permutations

data_path = "train.csv"
data = pd.read_csv(data_path)

target_thing = np.log1p(data["hours_spent"])

input_data = data.drop(columns=[
    "project_id",
    "hours_spent",
    "log_hours",
    "hours_bucket"
], errors="ignore")


qualitative_features = input_data.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

for col in qualitative_features:
    input_data[col] = (
        input_data[col]
        .fillna("Unknown")
        .astype(str)
    )

quantitative_features = input_data.select_dtypes(
    include=np.number
).columns.tolist()


input_training, input_validation, output_training, output_validation = (
    train_test_split(
        input_data,
        target_thing,
        test_size=0.2,
        random_state=42
    )
)


def evaluate(training, validation):

    validation = validation[training.columns]

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        verbose=False,
        early_stopping_rounds=100,
        random_seed=42
    )

    model.fit(
        training,
        output_training,
        cat_features=[
            c for c in qualitative_features
            if c in training.columns
        ],
        eval_set=(
            validation,
            output_validation
        )
    )

    predictions = np.einput_datapm1(
        model.predict(validation)
    )

    predictions = np.maximum(
        predictions,
        0
    )

    true_hours = np.expm1(
        output_validation
    )

    return RMSE(
        true_hours,
        predictions
    )

baseline = evaluate(
    input_training,
    input_validation
)

print("Baseline score:", baseline)


How_many_iterations = 250
learning_rate = 0.05
depth = 5

top_transformations = 15

results = []

single_operations = [
    "log1p",
    "signed_log",
    "sqrt",
    "square"
]


def apply_single_transform(
    dataframe,
    feature,
    operation
):

    result = dataframe.copy()

    x = dataframe[feature]

    if operation == "log1p":

        if (x.dropna() < 0).any():
            return None

        result[f"log1p_{feature}"] = np.log1p(
            x.clip(lower=0)
        )

    elif operation == "signed_log":

        result[f"signed_log_{feature}"] = (
            np.sign(x) *
            np.log1p(np.abs(x))
        )

    elif operation == "sqrt":

        if (x.dropna() < 0).any():
            return None

        result[f"sqrt_{feature}"] = np.sqrt(
            x.clip(lower=0)
        )

    elif operation == "square":

        result[f"square_{feature}"] = x ** 2

    else:

        raise ValueError(
            f"Unknown operation: {operation}"
        )

    return result

pair_operations = [
    "difference",
    "sum",
    "product",
    "absolute_difference"
]


def apply_pair_transform(
    dataframe,
    a,
    b,
    operation
):

    result = dataframe.copy()

    x = dataframe[a]
    y = dataframe[b]

    if operation == "difference":

        name = f"{a}_minus_{b}"

        result[name] = x - y

    elif operation == "sum":

        name = f"{a}_plus_{b}"

        result[name] = x + y

    elif operation == "product":

        name = f"{a}_times_{b}"

        result[name] = x * y

    elif operation == "absolute_difference":

        name = f"{a}_absdiff_{b}"

        result[name] = np.abs(x - y)

    else:

        raise ValueError(
            f"Unknown operation: {operation}"
        )

    return result, name


def screen(training, validation):

    validation = validation[training.columns]

    training = training.replace(
        [np.inf, -np.inf],
        np.nan
    )

    validation = validation.replace(
        [np.inf, -np.inf],
        np.nan
    )

    model = CatBoostRegressor(
        iterations=How_many_iterations,
        learning_rate=learning_rate,
        depth=depth,
        loss_function="RMSE",
        verbose=False,
        random_seed=42,
        early_stopping_rounds=30
    )

    model.fit(
        training,
        output_training,
        cat_features=[
            c for c in qualitative_features
            if c in training.columns
        ],
        eval_set=(
            validation,
            output_validation
        )
    )

    predictions = np.expm1(
        model.predict(validation)
    )

    predictions = np.maximum(
        predictions,
        0
    )

    true_hours = np.expm1(
        output_validation
    )

    score = RMSE(
        true_hours,
        predictions
    )

    improvement = baseline - score

    return score, improvement


print("SINGLE-FEATURE TRANSFORMATIONS")


single_count = (
    len(quantitative_features)
    * len(single_operations)
)

current = 0

for operation in single_operations:

    for feature in quantitative_features:

        current += 1

        name = f"{operation}_{feature}"

        print(
            f"\n[{current}/{single_count}] "
            f"Testing {name}"
        )

        training = apply_single_transform(
            input_training,
            feature,
            operation
        )

        validation = apply_single_transform(
            input_validation,
            feature,
            operation
        )

        if training is None or validation is None:

            print("Skipped")

            continue

        score, improvement = screen(
            training,
            validation
        )

        results.append({
            "feature": name,
            "operation": operation,
            "score": score,
            "improvement": improvement,
            "improvement_pct":
                improvement / baseline * 100
        })

        print(
            f"Score: {score:.6f} "
            f"({improvement:+.6f})"
        )



print("PAIRWISE TRANSFORMATIONS")


pair_count = (
    len(quantitative_features)
    * (len(quantitative_features) - 1)
    * len(pair_operations)
)

current = 0

for operation in pair_operations:

    for a, b in permutations(
        quantitative_features,
        2
    ):

        current += 1

        name = f"{a}_{operation}_{b}"

        print(
            f"\n[{current}/{pair_count}] "
            f"Testing {name}"
        )

        # IMPORTANT:
        # apply_pair_transform returns 2 things:
        # transformed dataframe + feature name

        training, name = apply_pair_transform(
            input_training,
            a,
            b,
            operation
        )

        validation, _ = apply_pair_transform(
            input_validation,
            a,
            b,
            operation
        )

        score, improvement = screen(
            training,
            validation
        )

        results.append({
            "feature": name,
            "operation": operation,
            "score": score,
            "improvement": improvement,
            "improvement_pct":
                improvement / baseline * 100
        })

        print(
            f"Score: {score:.6f} "
            f"({improvement:+.6f})"
        )

results = pd.DataFrame(results)

results = results.sort_values(
    "score",
    ascending=True
)

print("BEST TRANSFORMATIONS")

print(
    results.head(30).to_string(
        index=False
    )
)


selected = (
    results
    .head(top_transformations)
    ["feature"]
    .tolist()
)

print(
    f"\nSelected top "
    f"{top_transformations} transformations:"
)

for i, name in enumerate(
    selected,
    1
):

    print(
        f"{i}. {name}"
    )



print("FULL EVALUATION")


final_results = []


for i, name in enumerate(
    selected,
    1
):

    print(
        f"\n[{i}/{len(selected)}] "
        f"FULL TEST: {name}"
    )

    row = results[
        results["feature"] == name
    ].iloc[0]

    operation = row["operation"]

    training = input_training.copy()
    validation = input_validation.copy()

    if operation in single_operations:

        feature = name[
            len(operation) + 1:
        ]

        training = apply_single_transform(
            training,
            feature,
            operation
        )

        validation = apply_single_transform(
            validation,
            feature,
            operation
        )

    else:

        found = False

        for a, b in permutations(
            quantitative_features,
            2
        ):

            if operation == "difference":

                candidate = (
                    f"{a}_minus_{b}"
                )

            elif operation == "sum":

                candidate = (
                    f"{a}_plus_{b}"
                )

            elif operation == "product":

                candidate = (
                    f"{a}_times_{b}"
                )

            elif operation == "absolute_difference":

                candidate = (
                    f"{a}_absdiff_{b}"
                )

            else:

                candidate = None

            if candidate == name:

                training, _ = apply_pair_transform(
                    training,
                    a,
                    b,
                    operation
                )

                validation, _ = apply_pair_transform(
                    validation,
                    a,
                    b,
                    operation
                )

                found = True

                break

        if not found:

            print(
                "Could not reconstruct transformation"
            )

            continue

    score = evaluate(
        training,
        validation
    )

    improvement = (
        baseline - score
    )

    final_results.append({
        "feature": name,
        "operation": operation,
        "score": score,
        "improvement": improvement,
        "improvement_pct":
            improvement / baseline * 100
    })

    print(
        f"Score: {score:.6f} "
        f"({improvement:+.6f})"
    )

final_results = pd.DataFrame(
    final_results
)

final_results = final_results.sort_values(
    "score",
    ascending=True
)

print("FINAL RESULTS")

print(
    final_results.to_string(
        index=False
    )
)

if len(final_results) > 0:

    best = final_results.iloc[0]

    print("\n" + "=" * 60)
    print("BEST TRANSFORMATION")
    print("=" * 60)

    print(
        f"Feature:       {best['feature']}"
    )

    print(
        f"Operation:     {best['operation']}"
    )

    print(
        f"Score:         {best['score']:.6f}"
    )

    print(
        f"Baseline:      {baseline:.6f}"
    )

    print(
        f"Improvement:   {best['improvement']:+.6f}"
    )

    print(
        f"Improvement %: "
        f"{best['improvement_pct']:+.3f}%"
    )

BEST TRANSFORMATIONS
                                          feature           operation    score  improvement  improvement_pct
 num_commit_authors_minus_median_lines_per_commit          difference 0.659638     0.000045         0.006804
                lines_of_code_minus_max_dir_depth          difference 0.660119    -0.000437        -0.066181
            lines_of_code_absdiff_avg_file_length absolute_difference 0.660244    -0.000561        -0.085111
            avg_file_length_absdiff_lines_of_code absolute_difference 0.660244    -0.000561        -0.085111
  pct_code_in_first_commit_plus_lines_added_total                 sum 0.660308    -0.000626        -0.094844
  lines_added_total_plus_pct_code_in_first_commit                 sum 0.660308    -0.000626        -0.094844
              lines_of_code_minus_avg_file_length          difference 0.660544    -0.000862        -0.130684
         readme_num_headings_plus_num_directories                 sum 0.660622    -0.000940        -0.142460
         num_directories_plus_readme_num_headings                 sum 0.660622    -0.000940        -0.142460
            num_pull_requests_minus_lines_of_code          difference 0.660810    -0.001127        -0.170876
         lines_of_code_absdiff_num_3d_model_files absolute_difference 0.660868    -0.001185        -0.179668
         num_3d_model_files_absdiff_lines_of_code absolute_difference 0.660868    -0.001185        -0.179668
           num_test_files_plus_active_commit_days                 sum 0.660946    -0.001263        -0.191479
           active_commit_days_plus_num_test_files                 sum 0.660946    -0.001263        -0.191479
 num_3d_model_files_minus_pct_generic_commit_msgs          difference 0.661069    -0.001387        -0.210261
       lines_deleted_total_plus_project_span_days                 sum 0.661084    -0.001401        -0.212403
       project_span_days_plus_lines_deleted_total                 sum 0.661084    -0.001401        -0.212403
       lines_deleted_total_absdiff_num_test_files absolute_difference 0.661094    -0.001411        -0.213909
       num_test_files_absdiff_lines_deleted_total absolute_difference 0.661094    -0.001411        -0.213909
               lines_of_code_plus_num_image_files                 sum 0.661111    -0.001428        -0.216494
               num_image_files_plus_lines_of_code                 sum 0.661111    -0.001428        -0.216494
readme_num_headings_times_max_lines_in_one_commit             product 0.661179    -0.001497        -0.226943
max_lines_in_one_commit_times_readme_num_headings             product 0.661179    -0.001497        -0.226943
    num_image_files_times_max_lines_in_one_commit             product 0.661241    -0.001558        -0.236201
    max_lines_in_one_commit_times_num_image_files             product 0.661241    -0.001558        -0.236201
            lines_of_code_minus_project_span_days          difference 0.661270    -0.001587        -0.240612
         lines_of_code_minus_pct_primary_language          difference 0.661303    -0.001620        -0.245619
       max_dir_depth_minus_description_word_count          difference 0.661356    -0.001673        -0.253663
   pct_generic_commit_msgs_absdiff_num_test_files absolute_difference 0.661411    -0.001728        -0.261969
   num_test_files_absdiff_pct_generic_commit_msgs absolute_difference 0.661411    -0.001728        -0.261969

Selected top 15 transformations:
1. num_commit_authors_minus_median_lines_per_commit
2. lines_of_code_minus_max_dir_depth
3. lines_of_code_absdiff_avg_file_length
4. avg_file_length_absdiff_lines_of_code
5. pct_code_in_first_commit_plus_lines_added_total
6. lines_added_total_plus_pct_code_in_first_commit
7. lines_of_code_minus_avg_file_length
8. readme_num_headings_plus_num_directories
9. num_directories_plus_readme_num_headings
10. num_pull_requests_minus_lines_of_code
11. lines_of_code_absdiff_num_3d_model_files
12. num_3d_model_files_absdiff_lines_of_code
13. num_test_files_plus_active_commit_days
14. active_commit_days_plus_num_test_files
15. num_3d_model_files_minus_pct_generic_commit_msgs

FULL EVALUATION

[1/15] FULL TEST: num_commit_authors_minus_median_lines_per_commit
Score: 0.659043 (+0.000639)

[2/15] FULL TEST: lines_of_code_minus_max_dir_depth
Score: 0.659350 (+0.000332)

[3/15] FULL TEST: lines_of_code_absdiff_avg_file_length
Score: 0.657394 (+0.002288)

[4/15] FULL TEST: avg_file_length_absdiff_lines_of_code
Score: 0.657394 (+0.002288)

[5/15] FULL TEST: pct_code_in_first_commit_plus_lines_added_total
Score: 0.662666 (-0.002984)

[6/15] FULL TEST: lines_added_total_plus_pct_code_in_first_commit
Score: 0.662666 (-0.002984)

[7/15] FULL TEST: lines_of_code_minus_avg_file_length
Score: 0.654702 (+0.004981)

[8/15] FULL TEST: readme_num_headings_plus_num_directories
Score: 0.660467 (-0.000785)

[9/15] FULL TEST: num_directories_plus_readme_num_headings
Score: 0.660467 (-0.000785)

[10/15] FULL TEST: num_pull_requests_minus_lines_of_code
Score: 0.657749 (+0.001934)

[11/15] FULL TEST: lines_of_code_absdiff_num_3d_model_files
Score: 0.656707 (+0.002975)

[12/15] FULL TEST: num_3d_model_files_absdiff_lines_of_code
Score: 0.656707 (+0.002975)

[13/15] FULL TEST: num_test_files_plus_active_commit_days
Score: 0.659994 (-0.000312)

[14/15] FULL TEST: active_commit_days_plus_num_test_files
Score: 0.659994 (-0.000312)

[15/15] FULL TEST: num_3d_model_files_minus_pct_generic_commit_msgs
Score: 0.656545 (+0.003138)


FINAL RESULTS

                                         feature           operation    score  improvement  improvement_pct
             lines_of_code_minus_avg_file_length          difference 0.654702     0.004981         0.755021
num_3d_model_files_minus_pct_generic_commit_msgs          difference 0.656545     0.003138         0.475638
        lines_of_code_absdiff_num_3d_model_files absolute_difference 0.656707     0.002975         0.450980
        num_3d_model_files_absdiff_lines_of_code absolute_difference 0.656707     0.002975         0.450980
           lines_of_code_absdiff_avg_file_length absolute_difference 0.657394     0.002288         0.346885
           avg_file_length_absdiff_lines_of_code absolute_difference 0.657394     0.002288         0.346885
           num_pull_requests_minus_lines_of_code          difference 0.657749     0.001934         0.293126
num_commit_authors_minus_median_lines_per_commit          difference 0.659043     0.000639         0.096878
               lines_of_code_minus_max_dir_depth          difference 0.659350     0.000332         0.050401
          num_test_files_plus_active_commit_days                 sum 0.659994    -0.000312        -0.047245
          active_commit_days_plus_num_test_files                 sum 0.659994    -0.000312        -0.047245
        readme_num_headings_plus_num_directories                 sum 0.660467    -0.000785        -0.118983
        num_directories_plus_readme_num_headings                 sum 0.660467    -0.000785        -0.118983
 pct_code_in_first_commit_plus_lines_added_total                 sum 0.662666    -0.002984        -0.452305
 lines_added_total_plus_pct_code_in_first_commit                 sum 0.662666    -0.002984        -0.452305

BEST TRANSFORMATION

Feature:       lines_of_code_minus_avg_file_length
Operation:     difference
Score:         0.654702
Baseline:      0.659682
Improvement:   +0.004981
Improvement %: +0.755%

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_log_error


data = pd.read_csv("train.csv")

target_metric = "hours_spent"
seed = 42

y_original = data[target_metric].copy()
y = np.log1p(y_original)

X = data.drop(
    columns=[
        "project_id",
        "hours_spent",
        "log_hours",
        "hours_bucket"
    ],
    errors="ignore"
).copy()


cat_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

for col in cat_features:
    X[col] = (
        X[col]
        .fillna("Unknown")
        .astype(str)
    )


X_train, X_val, y_train, y_val, y_train_original, y_val_original = (
    train_test_split(
        X,
        y,
        y_original,
        test_size=0.2,
        random_state=seed
    )
)


def calculate_score(y_true, predictions):
    predictions = np.clip(predictions, 0, None)

    return root_mean_squared_log_error(
        y_true,
        predictions
    )

def add_behavior_features(df):

    df = df.copy()

    eps = 1e-6

    if {"num_commits", "active_commit_days"}.issubset(df.columns):

        df["commits_per_active_day"] = (
            df["num_commits"] /
            (df["active_commit_days"] + eps)
        )

    if {"num_commits", "project_span_days"}.issubset(df.columns):

        df["commits_per_project_day"] = (
            df["num_commits"] /
            (df["project_span_days"] + eps)
        )

    if {"active_commit_days", "project_span_days"}.issubset(df.columns):

        df["active_day_fraction"] = (
            df["active_commit_days"] /
            (df["project_span_days"] + eps)
        )

        df["inactive_days"] = (
            df["project_span_days"] -
            df["active_commit_days"]
        )

        df["inactive_day_fraction"] = (
            df["inactive_days"] /
            (df["project_span_days"] + eps)
        )

    if {"lines_of_code", "num_commits"}.issubset(df.columns):

        df["lines_per_commit"] = (
            df["lines_of_code"] /
            (df["num_commits"] + eps)
        )

    if {"lines_added_total", "num_commits"}.issubset(df.columns):

        df["lines_added_per_commit"] = (
            df["lines_added_total"] /
            (df["num_commits"] + eps)
        )

    if {"lines_deleted_total", "num_commits"}.issubset(df.columns):

        df["lines_deleted_per_commit"] = (
            df["lines_deleted_total"] /
            (df["num_commits"] + eps)
        )

    if {"num_commits", "num_commit_authors"}.issubset(df.columns):

        df["commits_per_author"] = (
            df["num_commits"] /
            (df["num_commit_authors"] + eps)
        )

    if {"lines_of_code", "num_commit_authors"}.issubset(df.columns):

        df["lines_per_author"] = (
            df["lines_of_code"] /
            (df["num_commit_authors"] + eps)
        )

    log_candidates = [
        "num_commits",
        "active_commit_days",
        "project_span_days",
        "lines_of_code",
        "lines_added_total",
        "lines_deleted_total",
        "num_commit_authors"
    ]

    for col in log_candidates:

        if col in df.columns:

            df[f"log1p_{col}"] = np.log1p(
                np.clip(df[col], 0, None)
            )

    df = df.replace([np.inf, -np.inf], np.nan)

    return df


print("\nAdding features...")

X_train = add_behavior_features(X_train)
X_val = add_behavior_features(X_val)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

configs = [


    (4, 0.03, 3),
    (5, 0.03, 3),
    (6, 0.03, 3),
    (7, 0.03, 3),
    (8, 0.03, 3),


    (5, 0.02, 3),
    (6, 0.02, 3),
    (7, 0.02, 3),

    (5, 0.05, 3),
    (6, 0.05, 3),
    (7, 0.05, 3),

    (6, 0.03, 1),
    (6, 0.03, 10),
    (6, 0.03, 30)
]

results = []

best_score = float("inf")
best_model = None
best_config = None

print("\nStarting hyperparameter search...")
print("=" * 70)


for i, (depth, lr, l2) in enumerate(configs, 1):

    print(
        f"\n[{i}/{len(configs)}] "
        f"Depth={depth}, LR={lr}, L2={l2}"
    )

    model = CatBoostRegressor(

        iterations=3000,

        depth=depth,

        learning_rate=lr,

        l2_leaf_reg=l2,

        loss_function="RMSE",

        random_seed=seed,

        random_strength=1,

        bootstrap_type="Bayesian",

        verbose=False,

        allow_writing_files=False,

        early_stopping_rounds=150
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_val, y_val),
        verbose=False
    )

    predictions_log = model.predict(X_val)

    predictions = np.expm1(predictions_log)

    predictions = np.clip(predictions, 0, None)

    score = calculate_score(
        y_val_original,
        predictions
    )

    best_iteration = model.get_best_iteration()

    results.append({

        "depth": depth,
        "learning_rate": lr,
        "l2_leaf_reg": l2,
        "best_iteration": best_iteration,
        "score": score

    })

    print(f"Score: {score:.6f}")
    print(f"Best iteration: {best_iteration}")

    if score < best_score:

        best_score = score
        best_model = model
        best_config = results[-1].copy()

        print("better model...")

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "score"
).reset_index(drop=True)

print(results_df.to_string(index=False))

print("BEST MODEL")

print(f"Score: {best_score:.6f}")
print(f"Depth: {best_config['depth']}")
print(f"Learning rate: {best_config['learning_rate']}")
print(f"L2: {best_config['l2_leaf_reg']}")
print(f"Best iteration: {best_config['best_iteration']}")

importance = pd.DataFrame({

    "feature": X_train.columns,

    "importance": best_model.get_feature_importance()

})

importance = importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("best predictors")

print(importance.head(40).to_string(index=False))

top_features = importance.head(25).sort_values(
    "importance"
)

baseline_score = 0.659682

improvement = baseline_score - best_score

improvement_pct = (
    improvement / baseline_score
) * 100

print("FINAL COMPARISON")

print(f"baseline: {baseline_score:.6f}")
print(f"new score: {best_score:.6f}")
print(f"improvment: {improvement_pct:.3f}%")
